01 - Validate Data

Goal of this notebook:
- load the raw AML dataset
- rename raw columns into canonical names
- validate required columns
- inspect missing values and data types
- generate a stable transaction_id
- save a cleaned parquet file for the next step

This notebook is meant to help me understand the data while executing each step.

# Importing Libraries and Setup Config

In [56]:
import pandas as pd
from pathlib import Path
import hashlib

## Env Var Setup

In [12]:
PROJECT_DIR = Path("..").resolve()
DATA_DIR = PROJECT_DIR / "data"

RAW_DIR = DATA_DIR / "raw"
RAW_FILE = RAW_DIR / "HI-Small_Trans.csv"

INTERIM_DIR =  DATA_DIR / "Interim"
INTERIM_FILE = INTERIM_DIR / "transactions_validated.parquet"

print(f"Project directory: {PROJECT_DIR} | Exists: {PROJECT_DIR.exists()}")
print(f"Raw file: {RAW_FILE} | Exists: {RAW_FILE.exists()}")
print(f"Interim file: {INTERIM_FILE} | Exists: {INTERIM_FILE.exists()}")

Project directory: C:\Users\surab\OneDrive\Documents\Personal\Projects\Fraud_Detection | Exists: True
Raw file: C:\Users\surab\OneDrive\Documents\Personal\Projects\Fraud_Detection\data\raw\HI-Small_Trans.csv | Exists: True
Interim file: C:\Users\surab\OneDrive\Documents\Personal\Projects\Fraud_Detection\data\Interim\transactions_validated.parquet | Exists: False


# Load the raw dataset and Inspect basic

In [13]:
df_raw = pd.read_csv(RAW_FILE)
print(df_raw.head())

          Timestamp  From Bank    Account  To Bank  Account.1  \
0  2022/09/01 00:20         10  8000EBD30       10  8000EBD30   
1  2022/09/01 00:20       3208  8000F4580        1  8000F5340   
2  2022/09/01 00:00       3209  8000F4670     3209  8000F4670   
3  2022/09/01 00:02         12  8000F5030       12  8000F5030   
4  2022/09/01 00:06         10  8000F5200       10  8000F5200   

   Amount Received Receiving Currency  Amount Paid Payment Currency  \
0          3697.34          US Dollar      3697.34        US Dollar   
1             0.01          US Dollar         0.01        US Dollar   
2         14675.57          US Dollar     14675.57        US Dollar   
3          2806.97          US Dollar      2806.97        US Dollar   
4         36682.97          US Dollar     36682.97        US Dollar   

  Payment Format  Is Laundering  
0   Reinvestment              0  
1         Cheque              0  
2   Reinvestment              0  
3   Reinvestment              0  
4   Reinvest

In [14]:
print("Data Info:",df_raw.info())

<class 'pandas.DataFrame'>
RangeIndex: 5078345 entries, 0 to 5078344
Data columns (total 11 columns):
 #   Column              Dtype  
---  ------              -----  
 0   Timestamp           str    
 1   From Bank           int64  
 2   Account             str    
 3   To Bank             int64  
 4   Account.1           str    
 5   Amount Received     float64
 6   Receiving Currency  str    
 7   Amount Paid         float64
 8   Payment Currency    str    
 9   Payment Format      str    
 10  Is Laundering       int64  
dtypes: float64(2), int64(3), str(6)
memory usage: 699.6 MB
Data Info: None


In [15]:
print("Raw data shape:", df_raw.shape)

Raw data shape: (5078345, 11)


In [16]:
print(f"Columns: \n {df_raw.columns.tolist()}")

Columns: 
 ['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']


In [17]:
print("Data types:\n", df_raw.dtypes)

Data types:
 Timestamp                 str
From Bank               int64
Account                   str
To Bank                 int64
Account.1                 str
Amount Received       float64
Receiving Currency        str
Amount Paid           float64
Payment Currency          str
Payment Format            str
Is Laundering           int64
dtype: object


## Rename columns
To make the rest of the pipeline clean and consistent, I want to convert the raw dataset headers into a standard internal schema.

In [18]:
RAW_TO_CANONICAL = {
    "Timestamp": "transaction_timestamp",
    "From Bank": "from_bank_id",
    "Account": "from_account_id",
    "To Bank": "to_bank_id",
    "Account.1": "to_account_id",
    "Amount Received": "amount_received",
    "Receiving Currency": "receiving_currency",
    "Amount Paid": "amount_paid",
    "Payment Currency": "payment_currency",
    "Payment Format": "payment_format",
    "Is Laundering": "is_laundering",
}


In [19]:

REQUIRED_COLUMNS = [
    "transaction_timestamp",
    "from_bank_id",
    "from_account_id",
    "to_bank_id",
    "to_account_id",
    "amount_received",
    "receiving_currency",
    "amount_paid",
    "payment_currency",
    "payment_format",
    "is_laundering",
]

In [21]:
df = df_raw.rename(columns=RAW_TO_CANONICAL).copy()
print("Columns after renaming:\n", df.columns)

Columns after renaming:
 Index(['transaction_timestamp', 'from_bank_id', 'from_account_id',
       'to_bank_id', 'to_account_id', 'amount_received', 'receiving_currency',
       'amount_paid', 'payment_currency', 'payment_format', 'is_laundering'],
      dtype='str')


## Check required columns

In [22]:
missing_required = [col for col in REQUIRED_COLUMNS if col not in df.columns]
extra_columns = [col for col in df.columns if col not in REQUIRED_COLUMNS]

print("Missing required columns:", missing_required)
print("Extra columns:", extra_columns)

Missing required columns: []
Extra columns: []


# Type Validation

## Timestamps are actual datetimes

In [24]:
df["transaction_timestamp"].isna().sum()

np.int64(0)

In [ ]:
# Convert into datetime
df["transaction_timestamp"] = pd.to_datetime(df["transaction_timestamp"], errors = "coerce") #errors="coerce" safely converts bad timestamps into NaT instead of failing
print("transaction_timestamp dtype:", df["transaction_timestamp"].dtype)

transaction_timestamp dtype: datetime64[us]


In [33]:
df["transaction_timestamp"].head()


0   2022-09-01 00:20:00
1   2022-09-01 00:20:00
2   2022-09-01 00:00:00
3   2022-09-01 00:02:00
4   2022-09-01 00:06:00
Name: transaction_timestamp, dtype: datetime64[us]

transaction_timestamp dtype: datetime64[us]


## Amount columns are numeric

In [35]:
numeric_cols = ["amount_received", "amount_paid"]
for col in numeric_cols:
    df[col]=pd.to_numeric(df[col], errors="coerce") 
    print(f"{col} dtype:", df[col].dtype)
    print(f"{col} NaN count:", df[col].isna().sum())
    

amount_received dtype: float64
amount_received NaN count: 0
amount_paid dtype: float64
amount_paid NaN count: 0


## Categorical/id-like columns are strings

In [36]:
string_cols = [
    "from_bank_id",
    "from_account_id",
    "to_bank_id",
    "to_account_id",
    "receiving_currency",
    "payment_currency",
    "payment_format",
]


In [38]:
for col in string_cols:
    print(f"Before conversion - {col} dtype:", df[col].dtype)
    df[col] = df[col].astype(str).str.strip() # Convert to string and remove leading/trailing whitespace
    print(f"After conversion - {col} dtype:", df[col].dtype)
    print(f"{col} NaN count:", df[col].isna().sum())
    print("-------------------------------")

Before conversion - from_bank_id dtype: int64
After conversion - from_bank_id dtype: str
from_bank_id NaN count: 0
-------------------------------
Before conversion - from_account_id dtype: str
After conversion - from_account_id dtype: str
from_account_id NaN count: 0
-------------------------------
Before conversion - to_bank_id dtype: int64
After conversion - to_bank_id dtype: str
to_bank_id NaN count: 0
-------------------------------
Before conversion - to_account_id dtype: str
After conversion - to_account_id dtype: str
to_account_id NaN count: 0
-------------------------------
Before conversion - receiving_currency dtype: str
After conversion - receiving_currency dtype: str
receiving_currency NaN count: 0
-------------------------------
Before conversion - payment_currency dtype: str
After conversion - payment_currency dtype: str
payment_currency NaN count: 0
-------------------------------
Before conversion - payment_format dtype: str
After conversion - payment_format dtype: str

## Target column is a clean binary label

In [39]:
df["is_laundering"].isna().sum()

np.int64(0)

In [42]:
print("is_laundering dtype:", df["is_laundering"].dtype)
df["is_laundering"] = pd.to_numeric(df["is_laundering"], errors="coerce").astype("int64") 
print("is_laundering dtype after conversion:", df["is_laundering"].dtype)

is_laundering dtype: int64
is_laundering dtype after conversion: int64


In [45]:
invalid_values = df[~df["is_laundering"].isin([0, 1])]
print("Invalid values in is_laundering column:\n", invalid_values[["is_laundering"]])

Invalid values in is_laundering column:
 Empty DataFrame
Columns: [is_laundering]
Index: []


## Inspect missingness and basic validation summaries

In [47]:
print("Min timestamp:", df["transaction_timestamp"].min())
print("Max timestamp:", df["transaction_timestamp"].max())

Min timestamp: 2022-09-01 00:00:00
Max timestamp: 2022-09-18 16:18:00


# Target Distribution

In [52]:
target_dist = df["is_laundering"].value_counts(dropna=False).sort_index().to_frame("count")
print("is_laundering distribution:\n", target_dist)

is_laundering distribution:
                  count
is_laundering         
0              5073168
1                 5177


In [53]:
target_dist["pct"] = (target_dist["count"] / len(df) * 100).round(2)
print("is_laundering distribution with pct:\n", target_dist)

is_laundering distribution with pct:
                  count   pct
is_laundering               
0              5073168  99.9
1                 5177   0.1


# Create stable transaction_id
The raw dataset does not come with the internal transaction_id I want to use in the pipeline.

So I will create one from row content in a reproducible way.

In [54]:
id_source_cols = [
    "transaction_timestamp",
    "from_bank_id",
    "from_account_id",
    "to_bank_id",
    "to_account_id",
    "amount_received",
    "receiving_currency",
    "amount_paid",
    "payment_currency",
    "payment_format",
    "is_laundering",
]

In [55]:
signature_df = df[id_source_cols].copy()

In [59]:
for col in signature_df.columns:
    if pd.api.types.is_datetime64_any_dtype(signature_df[col]):
        signature_df[col] = signature_df[col].dt.strftime("%Y-%m-%d %H:%M:%S")
    signature_df[col] = signature_df[col].astype(str).fillna("<NA>").str.strip()

signature = signature_df.agg("||".join, axis=1)

In [60]:
base_hash = signature.apply(
    lambda x : hashlib.sha256(x.encode("utf-8")).hexdigest()[:16]
)


In [61]:
duplicates = base_hash.duplicated(keep=False)
num_duplicates = duplicates.sum()
print(f"Number of duplicate transactions based on signature: {num_duplicates}")

Number of duplicate transactions based on signature: 18


In [65]:
duplicate_index = signature.groupby(signature).cumcount().astype(str).str.zfill(3)



In [68]:
df.insert(0,"transaction_id", "txn_"+base_hash +"_"+duplicate_index)
print("Data with transaction_id:\n", df[["transaction_id", "transaction_timestamp"]].head())

Data with transaction_id:
              transaction_id transaction_timestamp
0  txn_829ef002c65392e7_000   2022-09-01 00:20:00
1  txn_074e10942a6f76c2_000   2022-09-01 00:20:00
2  txn_558b19130bafb8d4_000   2022-09-01 00:00:00
3  txn_a7af3dc87f0b38b0_000   2022-09-01 00:02:00
4  txn_4eaa10cdc35a22e9_000   2022-09-01 00:06:00


## Check

In [72]:
dup = df["transaction_id"].duplicated(keep = False).sum()
print(f"Number of duplicate transaction_ids: {dup}")

Number of duplicate transaction_ids: 0


# Final Data and Save

In [73]:
print("Validated data shape:", df.shape)

Validated data shape: (5078345, 12)


In [74]:
df.head()

,transaction_id,transaction_timestamp,from_bank_id,from_account_id,to_bank_id,to_account_id,amount_received,receiving_currency,amount_paid,payment_currency,payment_format,is_laundering
0,txn_829ef002c65392e7_000,2022-09-01 00:20:00,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,txn_074e10942a6f76c2_000,2022-09-01 00:20:00,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,txn_558b19130bafb8d4_000,2022-09-01 00:00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,txn_a7af3dc87f0b38b0_000,2022-09-01 00:02:00,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,txn_4eaa10cdc35a22e9_000,2022-09-01 00:06:00,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


## Save Validated Dataset

**Why Parquet?** \
Parquet stores data column‑by‑column instead of row‑by‑row, which makes it smaller on disk and much faster to read for analytics.

In [76]:
df.to_parquet(INTERIM_FILE, index=False)
print(f"Validated data saved to {INTERIM_FILE}")

Validated data saved to C:\Users\surab\OneDrive\Documents\Personal\Projects\Fraud_Detection\data\Interim\transactions_validated.parquet


In [ ]:
print("=" * 70)
print("VALIDATION SUMMARY")
print("=" * 70)
print("Rows:", len(df))
print("Columns:", df.shape[1])
print()
print("Invalid timestamp count:", df["transaction_timestamp"].apply(lambda x: not isinstance(x, pd.Timestamp)).sum())
print("Invalid target values:", (~df["is_laundering"].isin([0, 1])).sum())
print("Duplicate transaction_id count:", dup)
print()
print("Target distribution:")
print(df["is_laundering"].value_counts(dropna=False).sort_index())
print("=" * 70)

VALIDATION SUMMARY
Rows: 5078345
Columns: 12

Invalid timestamp count: 0
Invalid target values: 0
Duplicate transaction_id count: 0

Target distribution:
is_laundering
0    5073168
1       5177
Name: count, dtype: int64


# End